# German Card Index — Synthetic Data Generator

Generate synthetic card crop images with German corner indices (**B** for Bube/Jack, **D** for Dame/Queen) to augment the rank classifier training data.

- **B** (Bube) → saved to `J` class folder
- **D** (Dame) → saved to `Q` class folder
- **Output:** ~500 images per rank into `data/rank_cards/{J,Q}/`

Run this notebook **before** Notebook 02 (rank classifier training) so the synthetic images are included in the training set.

> **Runtime:** CPU is sufficient — no GPU needed for image generation.

In [ ]:
# Install dependencies
!pip install -q "Pillow>=10.1" numpy matplotlib

In [ ]:
import os
import math
import random
import numpy as np
from dataclasses import dataclass
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont, ImageFilter, ImageEnhance
import matplotlib.pyplot as plt

random.seed(42)
np.random.seed(42)

# ── Card model ────────────────────────────────────────────────────────────────
@dataclass
class Card:
    suit: str       # "♠" "♣" "♥" "♦"
    value: str      # rank letter, e.g. "B" or "D"
    size: float     # diagonal of card face in pixels
    position: tuple # (cx, cy) — center of card in canvas coordinates

CARD_ASPECT = (5, 7)   # width : height  (standard poker card proportions)
SUPERSAMPLE = 4        # render card face at 4× then downscale → smooth text
MIN_CANVAS  = 256      # output images are always at least this size

def card_dimensions(diagonal: float) -> tuple[int, int]:
    """Return (width, height) of card face given its diagonal."""
    wr, hr = CARD_ASPECT
    d = math.sqrt(wr**2 + hr**2)
    return int(diagonal * wr / d), int(diagonal * hr / d)

def make_random_card(value: str, suit: str,
                     min_diag: float = 80, max_diag: float = 150) -> tuple["Card", int]:
    """
    Return a Card with random size/position and the square canvas size.

    Canvas = max(MIN_CANVAS, 2–3× diagonal).
    Card center ≥ diagonal/2 from every edge → card stays fully visible at any rotation.
    """
    d = random.uniform(min_diag, max_diag)
    canvas_size = max(MIN_CANVAS, int(random.uniform(2.0, 3.0) * d))
    margin = d / 2
    cx = random.uniform(margin, canvas_size - margin)
    cy = random.uniform(margin, canvas_size - margin)
    return Card(suit=suit, value=value, size=d, position=(cx, cy)), canvas_size

# ── Constants ─────────────────────────────────────────────────────────────────
RANK_DIR = "data/rank_cards"
NUM_IMAGES_PER_RANK = 500

GERMAN_RANKS = {
    "B": "J",   # Bube → Jack folder
    "D": "Q",   # Dame → Queen folder
}

SUITS = {
    "♠": (0, 0, 0),
    "♣": (0, 0, 0),
    "♥": (200, 0, 0),
    "♦": (200, 0, 0),
}

# Canvas backgrounds simulate real table/surface colors — distinct from white card face
CANVAS_BG_COLORS = [
    (45,  90,  45),   # green felt
    (35,  70,  35),   # dark green felt
    (80,  55,  35),   # wood brown
    (110, 75,  50),   # light wood
    (150, 140, 130),  # grey felt
    (55,  55,  70),   # dark fabric
    (190, 175, 155),  # beige/tan
    (30,  30,  30),   # near-black surface
]

TEXT_COLORS = [
    (0,   0,   0),
    (20,  20,  20),
    (0,   0,  100),
]

print(f"Will generate {NUM_IMAGES_PER_RANK} images each for: {list(GERMAN_RANKS.keys())}")
print(f"Output: {RANK_DIR}/J/ and {RANK_DIR}/Q/")
print(f"Canvas size: ≥{MIN_CANVAS}px, card diagonal: 80–150px")

## 1. Font Discovery

Find available fonts on the system (Colab or local). Card rank letters need serif and sans-serif fonts at various weights to create visual diversity.

In [ ]:
# Discover available TrueType fonts for rendering rank letters
# On Colab, fonts live in /usr/share/fonts/; locally they vary by OS

def find_system_fonts():
    """Find .ttf font files on the system."""
    font_dirs = [
        "/usr/share/fonts",
        "/usr/local/share/fonts",
        "/System/Library/Fonts",         # macOS
        "C:\\Windows\\Fonts",            # Windows
    ]
    fonts = []
    for d in font_dirs:
        if os.path.isdir(d):
            for root, _, files in os.walk(d):
                for f in files:
                    if f.lower().endswith((".ttf", ".otf")):
                        fonts.append(os.path.join(root, f))
    return fonts

all_fonts = find_system_fonts()
print(f"Found {len(all_fonts)} fonts on system")

def font_renders_all_suits(font_path: str, size: int = 40) -> bool:
    """Return True only if font renders all four suit symbols as non-blank pixels."""
    try:
        font = ImageFont.truetype(font_path, size)
        for suit in ["♠", "♣", "♥", "♦"]:
            img = Image.new("RGB", (60, 60), "white")
            draw = ImageDraw.Draw(img)
            draw.text((5, 5), suit, font=font, fill="black")
            arr = np.array(img)
            # If all pixels are near-white the suit symbol wasn't rendered
            if arr.min() > 200:
                return False
        return True
    except Exception:
        return False

# Filter to fonts that can render B, D AND all four suit symbols
usable_fonts = []
for font_path in all_fonts:
    try:
        font = ImageFont.truetype(font_path, 40)
        img = Image.new("RGB", (80, 60), "white")
        draw = ImageDraw.Draw(img)
        draw.text((5, 5), "BD", font=font, fill="black")
        arr = np.array(img)
        if arr.min() > 200:
            continue  # B/D didn't render
        if font_renders_all_suits(font_path):
            usable_fonts.append(font_path)
    except Exception:
        continue

print(f"Usable fonts (render B, D, and all suit symbols): {len(usable_fonts)}")
for f in usable_fonts[:10]:
    print(f"  {f}")

# Fallback: if fewer than 3 fonts found, use PIL default font with size variations.
# Note: PIL default font may not render suit unicode symbols — suits will be omitted.
if len(usable_fonts) < 3:
    print("WARNING: Few unicode-capable fonts found. Will use PIL default font with size variations.")

## 2. Card Rendering

Render a single synthetic card crop: white/cream background, rank letter in upper-left and lower-right corners (mirrored), suit symbol beside the rank.

In [ ]:
def render_scene(
    card: Card,
    canvas_size: int,
    suit_color: tuple,
    font_path: str | None = None,
    canvas_bg: tuple = (45, 90, 45),
    text_color: tuple = (0, 0, 0),
    rotation: float = 0.0,
) -> Image.Image:
    """
    Render a card on a canvas.

    Card face is always white. Canvas background is a distinct surface color.
    Card face is rendered at SUPERSAMPLE× resolution then downscaled with LANCZOS
    for smooth text at small sizes.
    """
    cw, ch = card_dimensions(card.size)

    # ── Render card face at high resolution, then downscale ───────────────────
    ss = SUPERSAMPLE
    cw_ss, ch_ss = cw * ss, ch * ss
    font_size = max(10, int(ch_ss * 0.22))
    suit_font_size = max(8, int(font_size * 0.75))

    def load_font(path, size):
        if path:
            try:
                return ImageFont.truetype(path, size)
            except Exception:
                pass
        return ImageFont.load_default(size=size)

    rank_font = load_font(font_path, font_size)
    suit_font = load_font(font_path, suit_font_size)

    card_ss = Image.new("RGB", (cw_ss, ch_ss), (255, 255, 255))
    draw = ImageDraw.Draw(card_ss)

    # Upper-left pip
    rx, ry = 3 * ss, 2 * ss
    draw.text((rx, ry), card.value, font=rank_font, fill=text_color)
    draw.text((rx + ss, ry + font_size + ss), card.suit, font=suit_font, fill=suit_color)

    # Lower-right pip (rotated 180°)
    corner_w = cw_ss // 2
    corner_h = font_size + suit_font_size + 8 * ss
    corner = Image.new("RGB", (corner_w, corner_h), (255, 255, 255))
    cdraw = ImageDraw.Draw(corner)
    cdraw.text((3 * ss, ss), card.value, font=rank_font, fill=text_color)
    cdraw.text((4 * ss, font_size + 2 * ss), card.suit, font=suit_font, fill=suit_color)
    corner = corner.rotate(180, expand=False)
    card_ss.paste(corner, (cw_ss - corner_w - 2 * ss, ch_ss - corner_h - 2 * ss))

    # Card border
    draw.rectangle([(0, 0), (cw_ss - 1, ch_ss - 1)], outline=(180, 180, 180), width=2 * ss)

    # Downscale to target size
    card_img = card_ss.resize((cw, ch), Image.LANCZOS)

    # ── Rotate (fill corners with canvas bg so they blend in) ─────────────────
    if rotation != 0:
        card_img = card_img.rotate(
            rotation, expand=True, resample=Image.BILINEAR, fillcolor=canvas_bg
        )

    # ── Composite onto canvas ──────────────────────────────────────────────────
    canvas = Image.new("RGB", (canvas_size, canvas_size), canvas_bg)
    x0 = int(card.position[0] - card_img.width / 2)
    y0 = int(card.position[1] - card_img.height / 2)
    canvas.paste(card_img, (x0, y0))

    return canvas


# ── Quick test ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for col, (suit, color) in enumerate(SUITS.items()):
    for row, letter in enumerate(["B", "D"]):
        card, cs = make_random_card(letter, suit)
        bg = random.choice(CANVAS_BG_COLORS)
        img = render_scene(card, cs, color, canvas_bg=bg, rotation=random.uniform(-15, 15))
        axes[row][col].imshow(img)
        axes[row][col].set_title(f"{letter}{suit}  d={card.size:.0f}px")
        axes[row][col].axis("off")
plt.suptitle("Rendered scenes — white card on coloured canvas")
plt.tight_layout()
plt.show()

## 3. Augmentation Pipeline

Apply randomized transforms to each rendered card to create visual diversity: rotation, brightness/contrast jitter, noise, color tint, and background bleed.

In [ ]:
def augment_image(img: Image.Image) -> Image.Image:
    """Pixel-level augmentations on the rendered scene. Rotation is handled in render_scene."""
    img = ImageEnhance.Brightness(img).enhance(random.uniform(0.8, 1.2))
    img = ImageEnhance.Contrast(img).enhance(random.uniform(0.8, 1.2))

    if random.random() < 0.3:
        arr = np.array(img, dtype=np.float32)
        tint = np.array([random.uniform(-10, 10)] * 3)
        img = Image.fromarray(np.clip(arr + tint, 0, 255).astype(np.uint8))

    if random.random() < 0.5:
        arr = np.array(img, dtype=np.float32)
        noise = np.random.normal(0, random.uniform(1, 10), arr.shape)
        img = Image.fromarray(np.clip(arr + noise, 0, 255).astype(np.uint8))

    if random.random() < 0.2:
        img = img.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.5, 1.5)))

    return img


# ── Augmentation preview ───────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for row, (letter, folder) in enumerate(GERMAN_RANKS.items()):
    suit, color = random.choice(list(SUITS.items()))
    card, cs = make_random_card(letter, suit)
    bg = random.choice(CANVAS_BG_COLORS)
    base = render_scene(card, cs, color, canvas_bg=bg, rotation=random.uniform(-15, 15))
    axes[row][0].imshow(base)
    axes[row][0].set_title("Base")
    axes[row][0].axis("off")
    for i in range(1, 5):
        card_i, cs_i = make_random_card(letter, suit)
        bg_i = random.choice(CANVAS_BG_COLORS)
        scene = render_scene(card_i, cs_i, color, canvas_bg=bg_i, rotation=random.uniform(-15, 15))
        axes[row][i].imshow(augment_image(scene))
        axes[row][i].set_title(f"Aug {i}")
        axes[row][i].axis("off")

plt.suptitle("Augmentation examples (B top, D bottom)")
plt.tight_layout()
plt.show()

## 4. Generate Synthetic Cards

Generate ~500 images per German rank (B→J folder, D→Q folder). Each image uses a random combination of font, suit, colors, and augmentations.

In [ ]:
suits_list = list(SUITS.items())

for german_letter, target_folder in GERMAN_RANKS.items():
    out_dir = os.path.join(RANK_DIR, target_folder)
    os.makedirs(out_dir, exist_ok=True)

    existing = len([f for f in os.listdir(out_dir) if f.endswith((".jpg", ".jpeg", ".png"))])
    print(f"\n{german_letter} → {target_folder}/: {existing} existing images")

    num_bases = 50
    augs_per_base = NUM_IMAGES_PER_RANK // num_bases  # 10
    generated = 0

    for base_idx in range(num_bases):
        suit, suit_color = suits_list[base_idx % len(suits_list)]
        text_color = random.choice(TEXT_COLORS)
        font_path = random.choice(usable_fonts) if usable_fonts else None

        for aug_idx in range(augs_per_base):
            card, canvas_size = make_random_card(german_letter, suit)
            canvas_bg = random.choice(CANVAS_BG_COLORS)
            rotation = random.uniform(-15, 15)
            scene = render_scene(card, canvas_size, suit_color,
                                 font_path=font_path, canvas_bg=canvas_bg,
                                 text_color=text_color, rotation=rotation)
            scene = augment_image(scene)
            filename = f"synth_{german_letter}_{base_idx:03d}_{aug_idx:02d}.jpg"
            scene.save(os.path.join(out_dir, filename), quality=90)
            generated += 1

    print(f"  Generated {generated} synthetic images → {out_dir}/")
    total_now = len([f for f in os.listdir(out_dir) if f.endswith((".jpg", ".jpeg", ".png"))])
    print(f"  Total in folder: {total_now}")

## 5. Visual QA

Display a grid of random synthetic samples from each rank to verify quality.

In [ ]:
# Display random samples of generated images
fig, axes = plt.subplots(2, 8, figsize=(20, 6))

for row, (german_letter, target_folder) in enumerate(GERMAN_RANKS.items()):
    out_dir = os.path.join(RANK_DIR, target_folder)
    synth_files = sorted([
        f for f in os.listdir(out_dir)
        if f.startswith(f"synth_{german_letter}")
    ])
    samples = random.sample(synth_files, min(8, len(synth_files)))

    for col, fname in enumerate(samples):
        img = Image.open(os.path.join(out_dir, fname))
        axes[row][col].imshow(img)
        axes[row][col].axis("off")
        if col == 0:
            axes[row][col].set_ylabel(f"{german_letter}→{target_folder}", fontsize=14)

plt.suptitle("Synthetic German Card Samples (B→J top, D→Q bottom)", fontsize=14)
plt.tight_layout()
plt.show()

## 6. Summary

Print final image counts per rank folder to confirm the synthetic data was added correctly.

In [ ]:
# Final summary: image counts per rank folder
print("Rank folder image counts:")
print("-" * 40)
total = 0
for rank in sorted(os.listdir(RANK_DIR)):
    rank_path = os.path.join(RANK_DIR, rank)
    if not os.path.isdir(rank_path):
        continue
    count = len([f for f in os.listdir(rank_path) if f.endswith((".jpg", ".jpeg", ".png"))])
    synth_count = len([f for f in os.listdir(rank_path) if f.startswith("synth_")])
    marker = " ← augmented" if synth_count > 0 else ""
    print(f"  {rank:>3}: {count:5d} images ({synth_count} synthetic){marker}")
    total += 1

print("-" * 40)
print(f"Total rank classes: {total}")
print("\nDone! Run Notebook 02 to train the rank classifier with German card data included.")